# Проверка сервисов BM25 и HNSW

Два отдельных сервиса, каждый отвечает на HTTP-запросы списком треков (`ids`) с оценками (`scores`):

* **BM25** (порт 8001) — поиск по словам: жанры, теги Last.fm, артист + название. Слова **английские**, как в каталоге.
* **HNSW** (порт 8002) — похожие по звучанию (эмбеддинги MuQ) на заданные треки.

Сами они ничего не рекомендуют: это кандидаты, из которых пайплайн собирает ответ (`inference.ipynb`).
Описание API: `docs/candgen_api.md`.

**Запуск локально** (в терминале, из корня репозитория; остановка — Ctrl+C):
```
BM25_PYTHON=.venv-candgen/bin/python HNSW_PYTHON=.venv-candgen/bin/python sh deploy/run_local.sh data/tracks_meta.parquet
```
На VPS сервисы уже запущены — нужны только адреса. Те же запросы можно делать в браузере: http://localhost:8001/docs

## Настройки

`DATA_DIR` — где лежит `tracks_meta`, из которого строились индексы (отсюда берутся названия треков).

In [ ]:
# ${X:-по умолчанию}: значение берётся из .env / окружения (сервер), иначе локальный run_local.sh
BM25_URL = "${BM25_URL:-http://127.0.0.1:8001}"
HNSW_URL = "${HNSW_URL:-http://127.0.0.1:8002}"
DATA_DIR = "data"

In [ ]:
import os
import pandas as pd

if os.path.basename(os.getcwd()) == "examples":
    os.chdir("..")
from recsys.data.crs import parquet_files
from recsys.config import load_env
from recsys.retrieval.remote import CandgenClient

load_env()  # адреса и ключи из .env
bm25 = CandgenClient(BM25_URL, api_key="${BM25_API_KEY}")
hnsw = CandgenClient(HNSW_URL, api_key="${HNSW_API_KEY}")
print("BM25:", bm25.url, "| HNSW:", hnsw.url)
columns = ["m4a_id", "m4a_artist", "m4a_song", "m4a_genres_full"]
tracks = pd.concat(pd.read_parquet(f, columns=columns) for f in parquet_files(DATA_DIR, "tracks_meta")).set_index("m4a_id")


def show(answer):
    """Ответ сервиса -> таблица: артист, песня, жанры, оценка."""
    return tracks.reindex(answer["ids"]).assign(score=answer["scores"])

## 1. Сервисы работают?

Какие индексы загружены и сколько в них треков. Ошибка `Connection refused` — сервис не запущен.

In [ ]:
pd.DataFrame(bm25.get("/health")["indexes"]).T

In [ ]:
pd.DataFrame(hnsw.get("/hnsw/health")["indexes"]).T

## 2. BM25: поиск по словам

* `words` — английские слова, регистр не важен
* `k` — сколько треков вернуть (до 1000)
* `index` — где искать: `genres` (по умолчанию), `tags`, `title` (артист + название; на сервере пока нет — ответ 404)

Чем больше слов запроса у трека, тем выше оценка.

In [ ]:
show(bm25.post("/bm25/search", {"words": ["rock"], "k": 5, "index": "tags"}))

In [ ]:
show(bm25.post("/bm25/search", {"words": ["hard", "rock"], "k": 5, "index": "genres"}))

In [ ]:
show(bm25.post("/bm25/search", {"words": ["nickelback"], "k": 5, "index": "title"}))

`exclude_ids` — какие треки не возвращать (пайплайн передаёт сюда уже показанные).

In [ ]:
first = bm25.post("/bm25/search", {"words": ["rock"], "k": 5, "index": "tags"})["ids"][0]
show(bm25.post("/bm25/search", {"words": ["rock"], "k": 5, "index": "tags", "exclude_ids": [first]}))

Русские слова не находятся: в каталоге английские теги. Перевод «рок» → `rock` делает пайплайн
при разборе запроса (`inference.ipynb`, раздел 5).

In [ ]:
bm25.post("/bm25/search", {"words": ["рок"], "k": 5, "index": "tags"})

## 3. HNSW: похожие по звучанию

На сервере HNSW пока нет: раздел работает, если запустить его локально
(`cd hnsw && HNSW_INDEXES_DIR=../indexes/hnsw ../.venv-candgen/bin/uvicorn service:app --port 8002`), иначе — `Connection refused`.

`track_ids` — треки, на которые должно быть похоже; `weights` — их веса (как история пользователя).
Сами треки запроса лучше исключить, иначе первым придёт тот же трек с оценкой 1.0. Оценка — сходство от −1 до 1.

In [ ]:
seed = tracks.index[1]
print("похожие на:", tracks.loc[seed, "m4a_artist"], "—", tracks.loc[seed, "m4a_song"])
show(hnsw.post("/hnsw/search", {"track_ids": [seed], "k": 5, "exclude_ids": [seed]}))

In [ ]:
seeds = list(tracks.index[:2])
show(hnsw.post("/hnsw/search", {"track_ids": seeds, "weights": [2, 1], "k": 5, "exclude_ids": seeds}))

## 4. Ошибки

| ответ | причина |
|---|---|
| `HTTP 404 Unknown index` | нет такого индекса |
| `HTTP 422` | ошибка в теле запроса: лишнее поле, `k` > 1000, у HNSW нет `track_ids` |
| `Connection refused`, таймаут | сервис не запущен или недоступен по сети |

In [ ]:
from recsys.retrieval.remote import CandgenError

try:
    bm25.post("/bm25/search", {"words": ["rock"], "k": 5, "index": "lyrics"})
except CandgenError as e:
    print(e)